# Fine-Tune SAM-2.1 on the spots dataset (local GPU, Windows)

Adapted from the Roboflow Colab notebook ([blog post](https://blog.roboflow.com/sam-2-1-fine-tuning)) to run locally:

- No Google Drive / Colab: all paths are relative to this notebook's folder (`scripts/01_model_training`).
- Dataset is read from `all_data/datasets/sam2_spots` (already exported from Roboflow in SAM-2 format).
- SAM-2 repo is the project-wide clone in `scripts/third_party/sam2` (shared with `02_model_inference/SAM2_spots_segmentation.ipynb`; it is outside this folder, so it does not shadow the `sam2` Python package).
- Installed with `SAM2_BUILD_CUDA=0` (the optional CUDA post-processing extension doesn't build on Windows and isn't needed).
- `torch.distributed` is disabled (`backend: none`, via a small patch to SAM-2's trainer): `nccl` is Linux-only and `gloo` is broken on Windows in recent PyTorch. Not needed for one GPU.
- Fewer dataloader workers (10 -> 4).

Run the cells top to bottom. **Restart the kernel after the install cell** the first time you run it.

## 1. Paths and GPU check

In [ ]:
import os
import sys
import subprocess
import torch

NB_DIR = os.path.abspath(os.getcwd())
ALL_DATA = os.path.abspath(os.path.join(NB_DIR, "..", ".."))
SAM2_DIR = os.path.join(ALL_DATA, "scripts", "third_party", "sam2")
DATA_DIR = os.path.join(ALL_DATA, "datasets", "sam2_spots")
WEIGHTS_DIR = os.path.join(ALL_DATA, "model_weights")
CKPT_DIR = os.path.join(SAM2_DIR, "checkpoints")
BASE_CKPT = os.path.join(CKPT_DIR, "sam2.1_hiera_base_plus.pt")
LOG_DIR = os.path.join(NB_DIR, "sam2_logs", "spots")
TRAIN_CFG = os.path.join(SAM2_DIR, "sam2", "configs", "train.yaml")

assert os.path.isdir(DATA_DIR), f"Dataset not found: {DATA_DIR}"
print("Python :", sys.executable)
print("Torch  :", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU    :", props.name, f"| {props.total_memory / 1024**3:.1f} GB | compute {props.major}.{props.minor}")
    print("bf16   :", torch.cuda.is_bf16_supported())

## 2. Clone and install SAM-2.1

Only needs to run once. Restart the kernel afterwards so `import sam2` picks up the new install.

In [ ]:
if not os.path.isdir(SAM2_DIR):
    subprocess.run(["git", "clone", "https://github.com/facebookresearch/sam2.git", SAM2_DIR], check=True)
else:
    print("SAM-2 repo already present:", SAM2_DIR)

In [ ]:
env = {**os.environ, "SAM2_BUILD_CUDA": "0"}
subprocess.run([sys.executable, "-m", "pip", "install", "--no-build-isolation", "-e", SAM2_DIR + "[dev]"],
               env=env, check=True)

### Patch SAM-2 training for single-GPU Windows

SAM-2's trainer always starts `torch.distributed` and wraps the model in DDP. On Windows `nccl` doesn't exist and
`gloo` fails in recent PyTorch builds (`makeDeviceForHostname(): unsupported gloo device`).
For a single GPU none of that is needed, so this cell adds a `backend: none` option that skips it:
no process group, no DDP wrapper, and a plain (non-distributed) sampler. Training math is identical for 1 GPU.
Safe to re-run (already-patched files are skipped).

In [ ]:
def patch_file(rel_path, replacements):
    path = os.path.join(SAM2_DIR, rel_path)
    with open(path, encoding="utf-8") as f:
        src = f.read()
    for old, new in replacements:
        if new in src:
            continue
        assert old in src, f"Patch target not found in {rel_path}:\n{old}"
        src = src.replace(old, new, 1)
    with open(path, "w", encoding="utf-8") as f:
        f.write(src)
    print("patched", rel_path)

patch_file("training/utils/train_utils.py", [(
    'def setup_distributed_backend(backend, timeout_mins):\n',
    'def setup_distributed_backend(backend, timeout_mins):\n'
    '    if backend == "none":  # single-GPU, no torch.distributed (local Windows patch)\n'
    '        logging.info("Distributed backend disabled (backend=none)")\n'
    '        return 0\n',
)])

patch_file("training/trainer.py", [
    ('        assert (\n            is_dist_avail_and_initialized()\n        ), "Torch distributed needs',
     '        assert (\n            distributed.backend == "none" or is_dist_avail_and_initialized()\n        ), "Torch distributed needs'),
    ('        assert isinstance(self.model, torch.nn.Module)\n\n        self.model = nn.parallel.DistributedDataParallel(',
     '        assert isinstance(self.model, torch.nn.Module)\n'
     '        if not is_dist_avail_and_initialized():  # backend=none: plain single-GPU model\n'
     '            return\n\n'
     '        self.model = nn.parallel.DistributedDataParallel('),
    ('                dist.barrier()\n', '                barrier()\n'),
])

patch_file("training/dataset/sam2_datasets.py", [
    ('            sampler = DistributedSampler(dataset, shuffle=self.shuffle)\n',
     '            from training.utils.distributed import get_rank, get_world_size\n'
     '            sampler = DistributedSampler(\n'
     '                dataset, num_replicas=get_world_size(), rank=get_rank(), shuffle=self.shuffle\n'
     '            )\n'),
])

## 3. Download the base SAM-2.1 checkpoint

`download_ckpts.sh` is a bash script, so download the Hiera-B+ checkpoint directly.

In [ ]:
import urllib.request

os.makedirs(CKPT_DIR, exist_ok=True)
url = "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_base_plus.pt"
if not os.path.isfile(BASE_CKPT):
    print("Downloading", url)
    urllib.request.urlretrieve(url, BASE_CKPT)
print(BASE_CKPT, f"{os.path.getsize(BASE_CKPT) / 1e6:.0f} MB")

## 4. Make dataset file names SAM-2.1 compatible

The SA-1B loader takes everything before the **first** dot as the image name and expects it to end in `_<int>`.
Roboflow names like `P4250031_jpg.rf.<hash>.jpg` become `P4250031_jpg_rf_<hash>_1.jpg` (image and its `.json` together).
Safe to re-run: already renamed files are left alone.

In [ ]:
import re

for split in ["train", "valid", "test"]:
    folder = os.path.join(DATA_DIR, split)
    if not os.path.isdir(folder):
        continue
    n = 0
    for filename in os.listdir(folder):
        if not filename.lower().endswith((".jpg", ".json")):
            continue
        new_filename = filename.replace(".", "_", filename.count(".") - 1)
        if not re.search(r"_\d+\.\w+$", new_filename):
            new_filename = new_filename.replace(".", "_1.")
        if new_filename != filename:
            os.rename(os.path.join(folder, filename), os.path.join(folder, new_filename))
            n += 1
    n_imgs = len([f for f in os.listdir(folder) if f.endswith(".jpg")])
    print(f"{split}: renamed {n} files, {n_imgs} images")

## 5. Write the training config

Same hyper-parameters as the Colab run (Hiera-B+, batch 1, 40 epochs, lr 5e-6 / 3e-6), with local paths, no distributed backend and 4 dataloader workers.
The Colab run peaked at ~7 GB GPU memory, so it fits in 12 GB. If you get CUDA out-of-memory, lower `max_num_objects` (e.g. 3 -> 2).
The `__PLACEHOLDERS__` are filled in with local paths when the file is written.

In [ ]:
NUM_EPOCHS = 40
NUM_WORKERS = 4

train_yaml = """
# @package _global_

scratch:
  resolution: 1024
  train_batch_size: 1
  num_train_workers: __NUM_WORKERS__
  num_frames: 1
  max_num_objects: 3
  base_lr: 5.0e-6
  vision_lr: 3.0e-06
  phases_per_epoch: 1
  num_epochs: __NUM_EPOCHS__

dataset:
  img_folder: "__TRAIN_DIR__"
  gt_folder: "__TRAIN_DIR__"
  multiplier: 2

vos:
  train_transforms:
    - _target_: training.dataset.transforms.ComposeAPI
      transforms:
        - _target_: training.dataset.transforms.RandomHorizontalFlip
          consistent_transform: True
        - _target_: training.dataset.transforms.RandomAffine
          degrees: 25
          shear: 20
          image_interpolation: bilinear
          consistent_transform: True
        - _target_: training.dataset.transforms.RandomResizeAPI
          sizes: ${scratch.resolution}
          square: true
          consistent_transform: True
        - _target_: training.dataset.transforms.ColorJitter
          consistent_transform: True
          brightness: 0.1
          contrast: 0.03
          saturation: 0.03
          hue: null
        - _target_: training.dataset.transforms.RandomGrayscale
          p: 0.05
          consistent_transform: True
        - _target_: training.dataset.transforms.ColorJitter
          consistent_transform: False
          brightness: 0.1
          contrast: 0.05
          saturation: 0.05
          hue: null
        - _target_: training.dataset.transforms.ToTensorAPI
        - _target_: training.dataset.transforms.NormalizeAPI
          mean: [0.485, 0.456, 0.406]
          std: [0.229, 0.224, 0.225]

trainer:
  _target_: training.trainer.Trainer
  mode: train_only
  max_epochs: ${times:${scratch.num_epochs},${scratch.phases_per_epoch}}
  accelerator: cuda
  seed_value: 123

  model:
    _target_: training.model.sam2.SAM2Train
    image_encoder:
      _target_: sam2.modeling.backbones.image_encoder.ImageEncoder
      scalp: 1
      trunk:
        _target_: sam2.modeling.backbones.hieradet.Hiera
        embed_dim: 112
        num_heads: 2
        drop_path_rate: 0.1
      neck:
        _target_: sam2.modeling.backbones.image_encoder.FpnNeck
        position_encoding:
          _target_: sam2.modeling.position_encoding.PositionEmbeddingSine
          num_pos_feats: 256
          normalize: true
          scale: null
          temperature: 10000
        d_model: 256
        backbone_channel_list: [896, 448, 224, 112]
        fpn_top_down_levels: [2, 3]
        fpn_interp_model: nearest

    memory_attention:
      _target_: sam2.modeling.memory_attention.MemoryAttention
      d_model: 256
      pos_enc_at_input: true
      layer:
        _target_: sam2.modeling.memory_attention.MemoryAttentionLayer
        activation: relu
        dim_feedforward: 2048
        dropout: 0.1
        pos_enc_at_attn: false
        self_attention:
          _target_: sam2.modeling.sam.transformer.RoPEAttention
          rope_theta: 10000.0
          feat_sizes: [32, 32]
          embedding_dim: 256
          num_heads: 1
          downsample_rate: 1
          dropout: 0.1
        d_model: 256
        pos_enc_at_cross_attn_keys: true
        pos_enc_at_cross_attn_queries: false
        cross_attention:
          _target_: sam2.modeling.sam.transformer.RoPEAttention
          rope_theta: 10000.0
          feat_sizes: [32, 32]
          rope_k_repeat: True
          embedding_dim: 256
          num_heads: 1
          downsample_rate: 1
          dropout: 0.1
          kv_in_dim: 64
      num_layers: 4

    memory_encoder:
      _target_: sam2.modeling.memory_encoder.MemoryEncoder
      out_dim: 64
      position_encoding:
        _target_: sam2.modeling.position_encoding.PositionEmbeddingSine
        num_pos_feats: 64
        normalize: true
        scale: null
        temperature: 10000
      mask_downsampler:
        _target_: sam2.modeling.memory_encoder.MaskDownSampler
        kernel_size: 3
        stride: 2
        padding: 1
      fuser:
        _target_: sam2.modeling.memory_encoder.Fuser
        layer:
          _target_: sam2.modeling.memory_encoder.CXBlock
          dim: 256
          kernel_size: 7
          padding: 3
          layer_scale_init_value: 1e-6
          use_dwconv: True
        num_layers: 2

    num_maskmem: 7
    image_size: ${scratch.resolution}
    sigmoid_scale_for_mem_enc: 20.0
    sigmoid_bias_for_mem_enc: -10.0
    use_mask_input_as_output_without_sam: true
    directly_add_no_mem_embed: true
    no_obj_embed_spatial: true
    use_high_res_features_in_sam: true
    multimask_output_in_sam: true
    iou_prediction_use_sigmoid: True
    use_obj_ptrs_in_encoder: true
    add_tpos_enc_to_obj_ptrs: true
    proj_tpos_enc_in_obj_ptrs: true
    use_signed_tpos_enc_to_obj_ptrs: true
    only_obj_ptrs_in_the_past_for_eval: true
    pred_obj_scores: true
    pred_obj_scores_mlp: true
    fixed_no_obj_ptr: true
    multimask_output_for_tracking: true
    use_multimask_token_for_obj_ptr: true
    multimask_min_pt_num: 0
    multimask_max_pt_num: 1
    use_mlp_for_obj_ptr_proj: true
    prob_to_use_pt_input_for_train: 0.5
    prob_to_use_pt_input_for_eval: 0.0
    prob_to_use_box_input_for_train: 0.5
    prob_to_use_box_input_for_eval: 0.0
    prob_to_sample_from_gt_for_train: 0.1
    num_frames_to_correct_for_train: 2
    num_frames_to_correct_for_eval: 1
    rand_frames_to_correct_for_train: True
    add_all_frames_to_correct_as_cond: True
    num_init_cond_frames_for_train: 2
    rand_init_cond_frames_for_train: True
    num_correction_pt_per_frame: 7
    use_act_ckpt_iterative_pt_sampling: false
    num_init_cond_frames_for_eval: 1
    forward_backbone_per_frame_for_eval: True

  data:
    train:
      _target_: training.dataset.sam2_datasets.TorchTrainMixedDataset
      phases_per_epoch: ${scratch.phases_per_epoch}
      batch_sizes:
        - ${scratch.train_batch_size}
      datasets:
        - _target_: training.dataset.vos_dataset.VOSDataset
          transforms: ${vos.train_transforms}
          training: true
          video_dataset:
            _target_: training.dataset.vos_raw_dataset.SA1BRawDataset
            img_folder: ${dataset.img_folder}
            gt_folder: ${dataset.gt_folder}
          multiplier: ${dataset.multiplier}
          sampler:
            _target_: training.dataset.vos_sampler.RandomUniformSampler
            num_frames: 1
            max_num_objects: ${scratch.max_num_objects}
      shuffle: True
      num_workers: ${scratch.num_train_workers}
      pin_memory: True
      drop_last: True
      collate_fn:
        _target_: training.utils.data_utils.collate_fn
        _partial_: true
        dict_key: all

  optim:
    amp:
      enabled: True
      amp_dtype: bfloat16

    optimizer:
      _target_: torch.optim.AdamW

    gradient_clip:
      _target_: training.optimizer.GradientClipper
      max_norm: 0.1
      norm_type: 2

    param_group_modifiers:
      - _target_: training.optimizer.layer_decay_param_modifier
        _partial_: True
        layer_decay_value: 0.9
        apply_to: 'image_encoder.trunk'
        overrides:
          - pattern: '*pos_embed*'
            value: 1.0

    options:
      lr:
        - scheduler:
            _target_: fvcore.common.param_scheduler.CosineParamScheduler
            start_value: ${scratch.base_lr}
            end_value: ${divide:${scratch.base_lr},10}
        - scheduler:
            _target_: fvcore.common.param_scheduler.CosineParamScheduler
            start_value: ${scratch.vision_lr}
            end_value: ${divide:${scratch.vision_lr},10}
          param_names:
            - 'image_encoder.*'
      weight_decay:
        - scheduler:
            _target_: fvcore.common.param_scheduler.ConstantParamScheduler
            value: 0.1
        - scheduler:
            _target_: fvcore.common.param_scheduler.ConstantParamScheduler
            value: 0.0
          param_names:
            - '*bias*'
          module_cls_names: ['torch.nn.LayerNorm']

  loss:
    all:
      _target_: training.loss_fns.MultiStepMultiMasksAndIous
      weight_dict:
        loss_mask: 20
        loss_dice: 1
        loss_iou: 1
        loss_class: 1
      supervise_all_iou: true
      iou_use_l1_loss: true
      pred_obj_scores: true
      focal_gamma_obj_score: 0.0
      focal_alpha_obj_score: -1.0

  distributed:
    backend: none          # single GPU: skip torch.distributed (nccl/gloo don't work on Windows); needs the patch cell
    find_unused_parameters: True

  logging:
    tensorboard_writer:
      _target_: training.utils.logger.make_tensorboard_logger
      log_dir: ${launcher.experiment_log_dir}/tensorboard
      flush_secs: 120
      should_log: True
    log_dir: ${launcher.experiment_log_dir}/logs
    log_freq: 10

  checkpoint:
    save_dir: ${launcher.experiment_log_dir}/checkpoints
    save_freq: 0  # 0 = only the last checkpoint is kept
    model_weight_initializer:
      _partial_: True
      _target_: training.utils.checkpoint_utils.load_state_dict_into_model
      strict: True
      ignore_unexpected_keys: null
      ignore_missing_keys: null
      state_dict:
        _target_: training.utils.checkpoint_utils.load_checkpoint_and_apply_kernels
        checkpoint_path: "__BASE_CKPT__"
        ckpt_state_dict_keys: ['model']

launcher:
  num_nodes: 1
  gpus_per_node: 1
  experiment_log_dir: "__LOG_DIR__"

submitit:
  partition: null
  account: null
  qos: null
  cpus_per_task: 10
  use_cluster: false
  timeout_hour: 24
  name: null
  port_range: [10000, 65000]
"""

def yaml_path(path):
    return path.replace("\\", "/")

train_yaml = (train_yaml.lstrip()
              .replace("__NUM_WORKERS__", str(NUM_WORKERS))
              .replace("__NUM_EPOCHS__", str(NUM_EPOCHS))
              .replace("__TRAIN_DIR__", yaml_path(os.path.join(DATA_DIR, "train")))
              .replace("__BASE_CKPT__", yaml_path(BASE_CKPT))
              .replace("__LOG_DIR__", yaml_path(LOG_DIR)))

with open(TRAIN_CFG, "w", encoding="utf-8") as f:
    f.write(train_yaml)
print("Wrote", TRAIN_CFG)

## 6. Train

Runs `training/train.py` as a subprocess (from inside `sam2_repo`) and prints progress every 30 s.
On the RTX 3500 Ada laptop GPU: ~0.65 s/iteration, 1065 iterations/epoch, so ~11-12 min per epoch, **~7.5 h for 40 epochs** (Colab A100: ~1 h).
Full logs: `sam2_logs/spots/logs/log.txt` (trainer) and `sam2_logs/spots/stdout.txt` (console). Interrupting the kernel stops training.
If `sam2_logs/spots/checkpoints/checkpoint.pt` exists, training **resumes** from it; delete `sam2_logs/spots` to start fresh.

In [ ]:
import time

train_env = {**os.environ, "HYDRA_FULL_ERROR": "1", "PYTHONUNBUFFERED": "1", "PYTHONIOENCODING": "utf-8"}
cmd = [sys.executable, os.path.join("training", "train.py"),
       "-c", "configs/train.yaml", "--use-cluster", "0", "--num-gpus", "1"]

os.makedirs(LOG_DIR, exist_ok=True)
stdout_path = os.path.join(LOG_DIR, "stdout.txt")
train_log = os.path.join(LOG_DIR, "logs", "log.txt")
KEEP = ("Train Epoch", "Estimated time", "Losses and meters", "Saving checkpoint", "Error", "Traceback")

with open(stdout_path, "w", encoding="utf-8") as out:
    proc = subprocess.Popen(cmd, cwd=SAM2_DIR, env=train_env, stdout=out, stderr=subprocess.STDOUT)
    pos = os.path.getsize(train_log) if os.path.exists(train_log) else 0
    try:
        while True:
            done = proc.poll() is not None
            if os.path.exists(train_log):
                with open(train_log, encoding="utf-8", errors="replace") as f:
                    f.seek(pos)
                    new_text = f.read()
                    pos = f.tell()
                for line in new_text.splitlines():
                    if any(k in line for k in KEEP):
                        print(line[:220])
            if done:
                break
            time.sleep(30)
    except KeyboardInterrupt:
        proc.terminate()
        proc.wait()
        print("Training stopped by user.")

print("Exit code:", proc.returncode)
if proc.returncode not in (0, None):
    print("".join(open(stdout_path, encoding="utf-8", errors="replace").readlines()[-40:]))

Training curves in TensorBoard (or run `tensorboard --logdir sam2_logs/spots/tensorboard` in a terminal and open http://localhost:6006):

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "sam2_logs/spots/tensorboard"


Copy the fine-tuned checkpoint to `model_weights` under a new name (the existing `sam_checkpoint.pt` is not touched):

In [ ]:
import shutil

FINETUNED_CKPT = os.path.join(LOG_DIR, "checkpoints", "checkpoint.pt")
dst = os.path.join(WEIGHTS_DIR, "sam2.1_hiera_b+_spots_local.pt")
shutil.copy2(FINETUNED_CKPT, dst)
print("Copied to", dst)

## 7. Visualize model results

Compare the fine-tuned model with the base SAM-2.1 model in automatic mask generation mode on a random validation image.

In [ ]:
%pip install supervision -q


### Load SAM-2.1

In [ ]:
import random
import numpy as np
from PIL import Image
import supervision as sv
from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator

if torch.cuda.get_device_properties(0).major >= 8:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

model_cfg = "configs/sam2.1/sam2.1_hiera_b+.yaml"
FINETUNED_CKPT = os.path.join(LOG_DIR, "checkpoints", "checkpoint.pt")

sam2 = build_sam2(model_cfg, FINETUNED_CKPT, device="cuda")
mask_generator = SAM2AutomaticMaskGenerator(sam2)

sam2_base = build_sam2(model_cfg, BASE_CKPT, device="cuda")
mask_generator_base = SAM2AutomaticMaskGenerator(sam2_base)

### Run inference on an image in automatic mask generation mode

In [ ]:
valid_dir = os.path.join(DATA_DIR, "valid")
image_path = os.path.join(valid_dir, random.choice([f for f in os.listdir(valid_dir) if f.endswith(".jpg")]))
print(image_path)
opened_image = np.array(Image.open(image_path).convert("RGB"))

with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
    result = mask_generator.generate(opened_image)
    base_result = mask_generator_base.generate(opened_image)

mask_annotator = sv.MaskAnnotator(color_lookup=sv.ColorLookup.INDEX)

detections = sv.Detections.from_sam(sam_result=result)
annotated_image = mask_annotator.annotate(opened_image.copy(), detections=detections)

base_detections = sv.Detections.from_sam(sam_result=base_result)
base_annotated_image = mask_annotator.annotate(opened_image.copy(), detections=base_detections)

sv.plot_images_grid(images=[annotated_image, base_annotated_image],
                    titles=["Fine-Tuned SAM-2.1", "Base SAM-2.1"], grid_size=(1, 2))